# Buffered and final cross-acquisition figures

Prefers saved memberships; reconstruction must be identified. Final cross-acquisition plot shows all-source refit and held-out target.

Source: `gd_spatial_protocol_figures.py`.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
"""TWO FIGURES | SINGLE-PANEL BUFFERED SPLIT + TWO-PANEL CROSS-ACQUISITION | NO CSV INPUT.
Image filenames supply GPS coordinates. Saved image lists are reused if present;
otherwise the original deterministic notebook split procedure is reproduced.
Reconstructed membership is explicitly identified in the console and provenance.
"""
from pathlib import Path
from itertools import combinations
import hashlib
import json
import re
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

RGB_DIR = None  # Automatic discovery of the RGB directory used in the notebook.
OUTPUT_DIR = None
BUFFER_M = 24.0
TARGET_VAL = TARGET_TEST = 40
EXPECTED = {"train": 207, "validation": 40, "test": 41, "excluded_buffer": 112}
GPS_PATTERN = re.compile(r"lat_(-?\d+(?:\.\d+)?)_lon_(-?\d+(?:\.\d+)?)", re.I)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
STYLES = {
    "train": ("Train", "#0072B2", "o"),
    "validation": ("Validation", "#E69F00", "^"),
    "test": ("Test", "#009E73", "s"),
    "excluded_buffer": ("Excluded", "#7C8087", "x"),
}


def sample_id(value):
    # Do not use Path.stem on extension-free names ending in decimal altitude.
    return re.sub(r"\.(?:jpe?g|png|tiff?)$", "", str(value).replace("\\", "/").split("/")[-1], flags=re.I)

def validate_assignments(frame):
    frame = frame.copy()
    required = {"sample_name", "latitude", "longitude", "split"}
    if not required.issubset(frame.columns):
        raise ValueError(f"Missing columns: {sorted(required - set(frame.columns))}")
    frame["sample_name"] = frame["sample_name"].map(sample_id)
    frame["split"] = frame["split"].astype(str).str.strip()
    if frame["sample_name"].duplicated().any():
        raise ValueError("Duplicate image identifiers in saved assignments.")
    counts = frame["split"].value_counts().to_dict()
    if counts != EXPECTED:
        raise ValueError(f"Saved counts {counts} do not match {EXPECTED}.")
    for column, limit in (("latitude", 90), ("longitude", 180)):
        frame[column] = pd.to_numeric(frame[column], errors="raise")
        if not (np.isfinite(frame[column]) & frame[column].between(-limit, limit)).all():
            raise ValueError(f"Invalid {column} values.")
    # Confirm the saved GPS coordinates against coordinate-labelled filenames.
    gps_pattern = re.compile(r"lat_(-?\d+(?:\.\d+)?)_lon_(-?\d+(?:\.\d+)?)", re.I)
    for row in frame.itertuples():
        match = gps_pattern.search(row.sample_name)
        if match and not np.allclose([float(match[1]), float(match[2])],
                                     [row.latitude, row.longitude], rtol=0, atol=1e-8):
            raise ValueError(f"Filename/CSV GPS mismatch: {row.sample_name}")
    return frame.sort_values("sample_name").reset_index(drop=True)


def gps_records_from_rgb():
    preferred = [Path(p) for p in (
        "/kaggle/input/datasets/hemal7/gd-rgbms-db/RGB",
        "/kaggle/input/hemal7/gd-rgbms-db/RGB",
        "/kaggle/input/jarin19/rgbmsdb/RGB",
        "/kaggle/input/gd-rgbms-db/RGB", "/content/GD-RGBMS-DB/RGB")]
    if RGB_DIR is not None:
        preferred = [Path(RGB_DIR)]
    folders = [p for p in preferred if p.is_dir()]
    if not folders and RGB_DIR is None:
        for location in ("/kaggle/input", "/content"):
            parent = Path(location)
            if parent.exists():
                folders.extend(p for p in parent.rglob("RGB") if p.is_dir())
    choices = []
    for folder in folders:
        records = []
        for path in sorted(folder.rglob("*")):
            if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            match = GPS_PATTERN.search(path.stem)
            if match:
                records.append({"sample_name": path.stem, "latitude": float(match[1]),
                                "longitude": float(match[2]), "rgb_path": str(path)})
        if len(records) == 400 and len({r["sample_name"] for r in records}) == 400:
            choices.append((pd.DataFrame(records).sort_values("sample_name").reset_index(drop=True), folder))
    if not choices:
        raise FileNotFoundError("The 400 GPS-labelled RGB images were not found. Attach the original "
                                "GD-RGBMS-DB dataset; no split CSV or trained weights are needed.")
    signatures = {tuple(frame.sample_name) for frame, folder in choices}
    if len(signatures) > 1:
        raise ValueError("More than one different 400-image RGB dataset was found. Set RGB_DIR explicitly.")
    print("RGB folder:", choices[0][1])
    return choices[0][0]


def membership_from_lists(frame, train, validation, test, excluded=None):
    lists = {"train": [sample_id(x) for x in train],
             "validation": [sample_id(x) for x in validation],
             "test": [sample_id(x) for x in test]}
    if any(len(values) != EXPECTED[key] for key, values in lists.items()):
        raise ValueError("Not the expected 207/40/41 buffered image lists.")
    names = [name for values in lists.values() for name in values]
    universe = set(frame.sample_name)
    if len(set(names)) != len(names) or not set(names).issubset(universe):
        raise ValueError("Saved lists overlap or contain images outside this RGB dataset.")
    missing = universe - set(names)
    if excluded is not None and set(map(sample_id, excluded)) != missing:
        raise ValueError("Saved excluded list disagrees with retained subsets.")
    labels = {name: key for key, values in lists.items() for name in values}
    result = frame.copy()
    result["split"] = result.sample_name.map(labels).fillna("excluded_buffer")
    return validate_assignments(result)


def load_membership_from_gps():
    frame = gps_records_from_rgb()
    candidates = []
    # Recognize lists left in memory by the previous 24 m training cell.
    for keys in (("TRAIN_NAMES", "VAL_NAMES", "TEST_NAMES"),
                 ("train_names", "val_names", "test_names")):
        if all(key in globals() for key in keys):
            try:
                candidate = membership_from_lists(frame, *(globals()[key] for key in keys))
                candidates.append((candidate, "in-memory buffered image lists: " + ", ".join(keys)))
            except (ValueError, TypeError):
                pass  # Unrelated standard-split variables are not buffered lists.
    roots = [Path(p) for p in (
        "/kaggle/working/gd_spatial24m_gatedfusion_3seeds",
        "/content/gd_spatial24m_gatedfusion_3seeds",
        "/content/drive/MyDrive/gd_spatial24m_gatedfusion_3seeds")]
    previous_output = globals().get("ROOT_OUT")
    if isinstance(previous_output, (str, Path)):
        roots.insert(0, Path(previous_output))
    for folder in dict.fromkeys(roots):
        files = [folder / name for name in ("train_list.txt", "val_list.txt", "test_list.txt")]
        if not all(path.is_file() for path in files):
            continue
        values = [[line.strip() for line in path.read_text(encoding="utf-8-sig").splitlines()
                   if line.strip()] for path in files]
        excluded_path = folder / "excluded_buffer_list.txt"
        excluded = excluded_path.read_text().splitlines() if excluded_path.exists() else None
        candidate = membership_from_lists(frame, *values, excluded=excluded)
        candidates.append((candidate, "saved buffered image lists: " + str(folder)))
    if candidates:
        signatures = {tuple(candidate.split) for candidate, source in candidates}
        if len(signatures) != 1:
            raise ValueError("Conflicting buffered image lists found in memory/output folders. "
                             "Use the lists from the actual training run; no arbitrary choice was made.")
        result, source = candidates[0]
        print("Membership source:", source)
        return result, source, False
    print("Saved buffered image lists are absent. Reproducing the ORIGINAL notebook split procedure.")
    result = validate_assignments(reproduce_original_split(frame))
    print("Membership source: original notebook GPS procedure (reconstructed; saved-run lists unavailable).")
    return result, "original notebook cell 67: deterministic GPS split reconstruction", True


def reproduce_original_split(frame):
    df = frame.copy()
    EARTH_RADIUS_M = 6371000.0
    mean_latitude_rad = np.deg2rad(df['latitude'].mean())
    mean_longitude = df['longitude'].mean()
    mean_latitude = df['latitude'].mean()
    df['x_m'] = np.deg2rad(df['longitude'] - mean_longitude) * EARTH_RADIUS_M * np.cos(mean_latitude_rad)
    df['y_m'] = np.deg2rad(df['latitude'] - mean_latitude) * EARTH_RADIUS_M
    coordinates = df[['x_m', 'y_m']].to_numpy()
    distance_matrix = cdist(coordinates, coordinates)
    adjacency = (distance_matrix <= BUFFER_M) & (distance_matrix > 0)
    number_of_components, component_labels = connected_components(csr_matrix(adjacency.astype(np.uint8)), directed=False, return_labels=True)
    df['spatial_component_24m'] = component_labels
    component_sizes = df.groupby('spatial_component_24m').size().sort_values()
    print('\n24 m spatial components:')
    print(component_sizes.to_string())
    if number_of_components != 2:
        raise ValueError(f'Expected 2 major spatial components, but found {number_of_components}. Please send the printed output before proceeding.')

    def find_best_buffered_end_split(component_df, target_holdout, buffer_m=24.0, target_tolerance=3):
        """
        Selects one end of a spatial field as validation/test.
        The opposite side becomes training.
        Images inside the buffer between them are excluded.
        """
        component_points = component_df[['x_m', 'y_m']].to_numpy()
        n = len(component_df)
        best_result = None
        angles = np.linspace(0.0, 179.75, 1440)
        minimum_k = max(15, target_holdout - target_tolerance)
        maximum_k = min(n - 15, target_holdout + target_tolerance)
        for angle_degree in angles:
            angle_radian = np.deg2rad(angle_degree)
            direction = np.array([np.cos(angle_radian), np.sin(angle_radian)])
            projected_position = component_points @ direction
            sorted_indices = np.argsort(projected_position)
            for side in ['low', 'high']:
                for k in range(minimum_k, maximum_k + 1):
                    if side == 'low':
                        holdout_local = sorted_indices[:k]
                        holdout_boundary = projected_position[holdout_local].max()
                        train_local = np.where(projected_position > holdout_boundary + buffer_m)[0]
                        excluded_local = np.where((projected_position > holdout_boundary) & (projected_position <= holdout_boundary + buffer_m))[0]
                    else:
                        holdout_local = sorted_indices[-k:]
                        holdout_boundary = projected_position[holdout_local].min()
                        train_local = np.where(projected_position < holdout_boundary - buffer_m)[0]
                        excluded_local = np.where((projected_position >= holdout_boundary - buffer_m) & (projected_position < holdout_boundary))[0]
                    if len(train_local) == 0:
                        continue
                    minimum_distance = cdist(component_points[holdout_local], component_points[train_local]).min()
                    if minimum_distance < buffer_m:
                        continue
                    score = (len(holdout_local) + len(train_local), -abs(len(holdout_local) - target_holdout), minimum_distance)
                    if best_result is None or score > best_result['score']:
                        best_result = {'score': score, 'angle_degree': angle_degree, 'side': side, 'holdout_local': holdout_local, 'train_local': train_local, 'excluded_local': excluded_local, 'minimum_distance': minimum_distance}
        if best_result is None:
            raise RuntimeError('No valid buffered split was found for this component.')
        component_global_indices = component_df.index.to_numpy()
        best_result['holdout_global'] = component_global_indices[best_result['holdout_local']]
        best_result['train_global'] = component_global_indices[best_result['train_local']]
        best_result['excluded_global'] = component_global_indices[best_result['excluded_local']]
        return best_result
    sorted_components = component_sizes.index.tolist()
    validation_component = sorted_components[0]
    test_component = sorted_components[1]
    validation_result = find_best_buffered_end_split(df[df['spatial_component_24m'] == validation_component], target_holdout=TARGET_VAL, buffer_m=BUFFER_M)
    test_result = find_best_buffered_end_split(df[df['spatial_component_24m'] == test_component], target_holdout=TARGET_TEST, buffer_m=BUFFER_M)
    df['split'] = 'excluded_buffer'
    df.loc[validation_result['train_global'], 'split'] = 'train'
    df.loc[test_result['train_global'], 'split'] = 'train'
    df.loc[validation_result['holdout_global'], 'split'] = 'validation'
    df.loc[test_result['holdout_global'], 'split'] = 'test'
    return df


def prepare_geometry(frame):
    frame = frame.copy()
    radians = np.deg2rad(frame[["latitude", "longitude"]].to_numpy())
    latitude, longitude = radians[:, 0], radians[:, 1]
    dlat, dlon = latitude[:, None] - latitude, longitude[:, None] - longitude
    h = np.sin(dlat / 2)**2 + np.cos(latitude[:, None]) * np.cos(latitude) * np.sin(dlon / 2)**2
    distances = 2 * 6371000.0 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))
    # Same geographic groups as the cross-acquisition notebook: 50 m connectivity,
    # Group A = 217 images and Group B = 183. Split assignments remain unchanged.
    n_groups, labels = connected_components(csr_matrix(distances <= 50.0), directed=False)
    group_sizes = {int(k): int((labels == k).sum()) for k in np.unique(labels)}
    if n_groups != 2 or sorted(group_sizes.values()) != [183, 217]:
        raise ValueError(f"Expected GPS groups of 217 and 183 images; found {group_sizes}.")
    group_a = next(k for k, size in group_sizes.items() if size == 217)
    frame["acquisition_group"] = np.where(labels == group_a, "A", "B")
    lat0, lon0 = frame.latitude.mean(), frame.longitude.mean()
    frame["east_m"] = np.deg2rad(frame.longitude - lon0) * 6371000.0 * np.cos(np.deg2rad(lat0))
    frame["north_m"] = np.deg2rad(frame.latitude - lat0) * 6371000.0
    verification, nearest = [], {}
    for first, second in combinations(("train", "validation", "test"), 2):
        i, j = np.flatnonzero(frame.split == first), np.flatnonzero(frame.split == second)
        block = distances[np.ix_(i, j)]
        a, b = np.unravel_index(block.argmin(), block.shape)
        minimum = float(block[a, b])
        if minimum < BUFFER_M - 1e-6:
            raise ValueError(f"Saved split fails 24 m separation: {first}/{second} = {minimum:.3f} m.")
        verification.append({"comparison": f"{first} vs {second}",
                             "minimum_camera_centre_distance_m": minimum})
        nearest[(first, second)] = (int(i[a]), int(j[b]), minimum)
    return frame, pd.DataFrame(verification), nearest


# Match the second notebook's scatter-plot style and size.
BUFFERED_FIGSIZE = (11, 8)
CROSS_FIGSIZE = (17, 7.5)
DPI = 600


def style_axis(ax, frame):
    """Use the same local-metre scatter coordinates and light grid as notebook 2."""
    xspan = max(float(frame.east_m.max() - frame.east_m.min()), 1.0)
    yspan = max(float(frame.north_m.max() - frame.north_m.min()), 1.0)
    ax.set_xlim(frame.east_m.min()-.07*xspan, frame.east_m.max()+.07*xspan)
    ax.set_ylim(frame.north_m.min()-.12*yspan, frame.north_m.max()+.20*yspan)
    ax.set_xlabel('Local East–West position (m)', fontsize=12)
    ax.set_ylabel('Local North–South position (m)', fontsize=12)
    ax.tick_params(labelsize=10)
    ax.ticklabel_format(style='plain', useOffset=False)
    ax.grid(True, alpha=.23, linewidth=.65, zorder=0)
    for spine in ax.spines.values():
        spine.set_color('#B8BDC5')
    # As in notebook 2, axes use their labelled coordinate ranges; no footprint
    # polygons, metre-scale circles, or inferred buffer boundaries are drawn.


def mark_group(ax, subset, label):
    ax.annotate(label, xy=(subset.east_m.median(), subset.north_m.max()),
                xytext=(0, 13), textcoords='offset points', ha='center', va='bottom',
                fontsize=11, fontweight='bold', zorder=6,
                bbox={'facecolor':'white','edgecolor':'#D9DEE5','alpha':.95,'pad':3.5})


def buffered_figure(frame):
    """One panel containing all 400 locations and four original subset labels."""
    fig, ax=plt.subplots(figsize=BUFFERED_FIGSIZE)
    sizes={'train':48,'validation':80,'test':68,'excluded_buffer':55}
    for key in ('excluded_buffer','train','validation','test'):
        subset=frame[frame.split==key]
        label,color,marker=STYLES[key]
        if key=='excluded_buffer':
            options={'linewidths':1.35}
            label='Excluded (buffer)'
        else:
            options={'edgecolors':'white','linewidths':.5}
        ax.scatter(subset.east_m,subset.north_m,s=sizes[key],c=color,marker=marker,
                   alpha=.88 if key=='train' else .97,zorder=3 if key=='excluded_buffer' else 4,
                   label=f'{label} (n = {len(subset)})',**options)
    style_axis(ax,frame)
    for group,subset in frame.groupby('acquisition_group'):
        mark_group(ax,subset,f'Group {group}')
    ax.set_title('24 m spatially buffered split',fontsize=16,fontweight='bold',pad=17)
    handles,labels=ax.get_legend_handles_labels()
    order=[labels.index(next(label for label in labels if label.startswith(name)))
           for name in ('Train','Validation','Test','Excluded')]
    ax.legend([handles[i] for i in order],[labels[i] for i in order],
              loc='upper center',bbox_to_anchor=(.5,-.16),ncol=4,fontsize=10,
              frameon=False,columnspacing=1.3,handletextpad=.5,markerscale=1.15)
    fig.text(.5,.018,'GPS camera-centre locations; colours and markers indicate subset membership.',
             ha='center',fontsize=9,color='#50565F')
    fig.subplots_adjust(left=.11,right=.97,top=.90,bottom=.22)
    return fig


def cross_acquisition_figure(frame):
    """Show final source-group training and complete held-out target evaluation."""
    fig,axes=plt.subplots(1,2,figsize=CROSS_FIGSIZE,sharex=True,sharey=True)
    for ax,source,target,panel in zip(axes,('A','B'),('B','A'),('a','b')):
        training=frame[frame.acquisition_group==source]
        testing=frame[frame.acquisition_group==target]
        ax.scatter(training.east_m,training.north_m,s=49,c=STYLES['train'][1],marker='o',
                   alpha=.86,edgecolors='white',linewidths=.5,zorder=3,
                   label=f'Training: Group {source} (n = {len(training)})')
        ax.scatter(testing.east_m,testing.north_m,s=65,c=STYLES['test'][1],marker='s',
                   alpha=.94,edgecolors='white',linewidths=.5,zorder=4,
                   label=f'Unseen test: Group {target} (n = {len(testing)})')
        style_axis(ax,frame)
        for group,subset in frame.groupby('acquisition_group'):
            mark_group(ax,subset,f'Group {group}')
        ax.set_title(f'({panel}) Train on Group {source}; test on unseen Group {target}',
                     fontsize=12.5,fontweight='bold',pad=15)
        ax.legend(loc='upper center',bbox_to_anchor=(.5,-.16),ncol=2,fontsize=9.5,
                  frameon=False,columnspacing=1.,handletextpad=.5)
    axes[1].tick_params(labelleft=True)
    fig.text(.5,.022,'The target acquisition group is held out from model fitting, checkpoint selection, and threshold selection.',
             ha='center',fontsize=10,color='#50565F')
    fig.subplots_adjust(left=.065,right=.98,top=.90,bottom=.23,wspace=.23)
    return fig


def save_figure(fig, folder, name):
    for extension in ('png','pdf','svg'):
        fig.savefig(folder/f'{name}.{extension}',dpi=DPI if extension=='png' else 300,
                    bbox_inches='tight',facecolor='white')
    plt.show()
    plt.close(fig)


def main():
    frame,source,reconstructed=load_membership_from_gps()
    frame,verification,_=prepare_geometry(frame)
    work=Path('/kaggle/working') if Path('/kaggle/working').exists() else (
        Path('/content') if Path('/content').exists() else Path.cwd())
    out=Path(OUTPUT_DIR) if OUTPUT_DIR is not None else work/'gd_spatial_protocol_figures'
    out.mkdir(parents=True,exist_ok=True)
    with plt.rc_context({'font.family':'DejaVu Sans','pdf.fonttype':42,
                         'ps.fonttype':42,'svg.fonttype':'none'}):
        save_figure(buffered_figure(frame),out,'buffered24m_single_panel')
        save_figure(cross_acquisition_figure(frame),out,'cross_acquisition_two_panels')
    frame.to_csv(out/'plotted_image_membership.csv',index=False)
    verification.to_csv(out/'buffered_subset_minimum_distances.csv',index=False)
    counts=pd.crosstab(frame.acquisition_group,frame.split).reindex(
        index=['A','B'],columns=list(EXPECTED),fill_value=0)
    counts.to_csv(out/'group_and_subset_counts.csv')
    caption=('Spatial distribution of images in the 24 m buffered split. Colours and markers distinguish '
             'training, validation, test, and excluded images. Points show GPS camera-centre locations '
             'rather than image footprints.')
    if reconstructed:
        caption+=' Assignments were reproduced using the original deterministic notebook procedure.'
    (out/'buffered_caption.txt').write_text(caption,encoding='utf-8')
    (out/'cross_acquisition_caption.txt').write_text(
        'Bidirectional cross-acquisition evaluation: (a) training on Group A (217 images) and testing on '
        'unseen Group B (183 images); (b) training on Group B and testing on unseen Group A. Blue circles '
        'indicate source-group training images and green squares indicate held-out target images. '
        'Checkpoint and threshold selection used development data within the source group only; '
        'the panels show final source-group training and target-group evaluation.',encoding='utf-8')
    provenance={'source':source,'split_reconstructed':reconstructed,
                'matched_saved_training_membership':not reconstructed,'counts':EXPECTED,
                'group_counts':{'A':217,'B':183},'buffer_m':BUFFER_M,
                'buffered_panels':1,'cross_acquisition_panels':2,
                'coordinates':'local metre coordinates derived from RGB-filename GPS',
                'distance_verification':'haversine camera-centre separation; not footprint overlap',
                'plot_style_reference':'gd-second.ipynb, local-metre scatter plots (11 x 8 inches)'}
    (out/'figure_provenance.json').write_text(json.dumps(provenance,indent=2),encoding='utf-8')
    archive=out/'gd_spatial_protocol_figures.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(out.iterdir()):
            if path.is_file() and path.suffix in {'.png','.pdf','.svg','.csv','.txt','.json'}:
                bundle.write(path,path.name)
    print('\nGROUP / SUBSET COUNTS\n',counts.to_string())
    print('\nCAMERA-CENTRE DISTANCES\n',verification.to_string(index=False))
    print('\nFigure 1:',out/'buffered24m_single_panel.png')
    print('Figure 2:',out/'cross_acquisition_two_panels.png')
    print('Download:',archive)
    return out


if __name__=='__main__':
    main()
